In [1]:
import os
import shutil
import datetime 
import json 

In [2]:
def organize_files_into_folders(master_folder_path):
    """
    Creates a distinct subfolder for each file in the master folder,
    moves the file into it, and places a metadata.json file in the new folder.

    Args:
        master_folder_path (str): The path to the master folder containing the files.
    """
    if not os.path.isdir(master_folder_path):
        print(f"Error: Master folder '{master_folder_path}' not found.")
        print("Please ensure the specified folder exists.")
        return

    print(f"Organizing files and creating metadata in: {master_folder_path}\n")

    items_processed = 0
    items_skipped = 0
    
    files_to_organize = []
    for item_name in os.listdir(master_folder_path):
        item_path = os.path.join(master_folder_path, item_name)
        if os.path.isfile(item_path):
            files_to_organize.append(item_name)
        elif os.path.isdir(item_path):
            print(f"Skipping directory: '{item_name}' (already a folder)")
            items_skipped += 1
        else:
            print(f"Skipping unknown item type: '{item_name}'")
            items_skipped += 1

    for item_name in files_to_organize:
        item_path = os.path.join(master_folder_path, item_name)

        if not os.path.isfile(item_path):
            print(f"Skipping '{item_name}': No longer a file or already processed.")
            items_skipped += 1
            continue

        file_name_without_extension, _ = os.path.splitext(item_name)
        
        target_folder_name = file_name_without_extension
        final_folder_path = os.path.join(master_folder_path, target_folder_name)

        temp_folder_path = None 

        try:
            if os.path.isdir(final_folder_path):
                print(f"Skipping '{item_name}': A folder '{target_folder_name}' already exists.")
                items_skipped += 1
                continue
            
            # --- Conflict Resolution & Folder Creation ---
            temp_folder_name = f"__temp_organize_{target_folder_name}_{datetime.datetime.now().strftime('%Y%m%d%H%M%S%f')}__"
            temp_folder_path = os.path.join(master_folder_path, temp_folder_name)

            os.makedirs(temp_folder_path, exist_ok=True)
            print(f"Created temporary folder: {temp_folder_path}")

            temp_destination_path = os.path.join(temp_folder_path, item_name)
            shutil.move(item_path, temp_destination_path)
            print(f"Temporarily moved '{item_name}' to '{temp_folder_path}'")

            os.rename(temp_folder_path, final_folder_path)
            print(f"Renamed '{temp_folder_name}' to '{final_folder_path}'")
            print(f"Successfully organized '{item_name}' into '{final_folder_path}'")
            items_processed += 1

            # --- Create and Populate metadata.json ---
            metadata_file_path = os.path.join(final_folder_path, "metadata.json")
            
            # Define the structure for your metadata
            metadata = {
                "title": "",
                "author": "",
                "genre": "",
                "issued_date": "", # Format example: "YYYY-MM-DD"
                "written_date": "", # Format example: "YYYY-MM-DD"
                "ocr_confidence": ""
            }

            with open(metadata_file_path, 'w', encoding='utf-8') as f:
                json.dump(metadata, f, indent=4)
            print(f"Created '{metadata_file_path}' with placeholder metadata.")

        except Exception as e:
            print(f"Error processing '{item_name}': {e}")
            items_skipped += 1
            if temp_folder_path and os.path.exists(temp_folder_path):
                try:
                    if not os.listdir(temp_folder_path):
                        os.rmdir(temp_folder_path)
                        print(f"Cleaned up empty temporary folder: {temp_folder_path}")
                except OSError as cleanup_error:
                    print(f"Warning: Could not clean up temporary folder '{temp_folder_path}': {cleanup_error}")

    print(f"\n--- Organization and Metadata Creation Complete ---")
    print(f"Files processed: {items_processed}")
    print(f"Items skipped (directories, already processed, or errors): {items_skipped}")
    print(f"Check the '{master_folder_path}' folder to see the results, including new 'metadata.json' files.")


In [3]:
# --- Example Usage ---
if __name__ == "__main__":
    # IMPORTANT: Replace 'path/to/your/master_folder' with the actual path
    # where your files are located.
    # For demonstration, let's assume a 'master_files' folder exists
    # in the same directory as this script.

    # Get the current directory of the script
    current_directory = os.getcwd()
    
    # Define the path to your master folder.
    # If your files are directly in a folder named 'my_documents' in the same location as the script:
    # master_folder = os.path.join(current_directory, "my_documents")
    
    # If you want to use the 'data' folder mentioned previously:
    master_folder = os.path.join(current_directory, "data", "OCR_Final")

    # Before running, make sure to:
    # 1. Create the 'data' folder (or your chosen master folder).
    # 2. Place some files (e.g., text files, images, PDFs) directly inside it.
    #    DO NOT put subfolders in it if you want only files to be moved.

    organize_files_into_folders(master_folder)

Organizing files and creating metadata in: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/Surya_OCR/data/OCR_Final

Created temporary folder: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/Surya_OCR/data/OCR_Final/__temp_organize_ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව_20250726163423196294__
Temporarily moved 'ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව' to '/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/Surya_OCR/data/OCR_Final/__temp_organize_ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව_20250726163423196294__'
Renamed '__temp_organize_ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව_20250726163423196294__' to '/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/Surya_OCR/data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව'
Successfully organized 'ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව' into '/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/Surya_OCR/data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිව